# Model Development & Tracking 
Covers **Stage 2.2–2.4 + Stage 3.** Complete every `# TODO` in this notebook **and** in the `src/` modules it imports. 

- Each stage opens with a **sub-task checklist**
- Capture any repo-generated evidence (MLflow UI, Docker build, CI run, drift report) as screenshots/summaries **inside the notebook/report**.

**File ownership** — 
- *Provided:* `config.py`, `src/evaluate.py`. 
- *Provided to extend:* `src/model.py`, `src/train.py`, `app.py`. 
- *You build:* this notebook.

### 0. Setup

In [2]:
#set to reload python modules automatically when they are changed
%load_ext autoreload
%autoreload 2

In [3]:
import config
# TODO: imports (torch, mlflow, pandas, matplotlib) as needed
import torch
import mlflow
import pandas as pd
import matplotlib.pyplot as plt

from src.model import build_model, trainable_parameters

torch.manual_seed(config.RANDOM_SEED)

print("Device:", config.DEVICE)
print("Random seed:", config.RANDOM_SEED)
print("Number of classes:", config.NUM_CLASSES)
print("Freeze backbone:", config.FREEZE_BACKBONE)


Device: cpu
Random seed: 42
Number of classes: 2
Freeze backbone: True


## **Stage 2.2 — Transfer-Learning Model** <font color="red">[7 marks]</font>

- **2.2.1 — ImageNet-pretrained ResNet18, backbone frozen [4]**
- **2.2.2 — New 2-class head replaces fc (head trained) [3]**

**Objective:** Build an ImageNet-pretrained ResNet18 with a frozen backbone + new 2-class head.

**Implement in:** src/model.build_model

**Inputs → Outputs:** → a torch model; only the head's params have requires_grad=True

**TODO:** load weights=IMAGENET1K + freeze the backbone (2.2.1); replace fc with a 2-class Linear head so only the head trains (2.2.2); print total vs trainable params.

**Depends on:** Stage 2.1 transformations completed in Data_Preparation.ipynb  ·  **Document here:** the param counts (trainable should be ~the head only).

In [ ]:
# TODO 2.2.1-2.2.2: from src.model import build_model, trainable_parameters; build + print params

# Build the ImageNet-pretrained ResNet18 model.
torch.manual_seed(config.RANDOM_SEED)

# load the model customized in src/model.py
model = build_model()

# Count the total number of parameters and the number of trainable parameters in the model.
total_parameter_count = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_parameter_count = sum(
    parameter.numel()
    for parameter in trainable_parameters(model)
)

# Print the model backbone, final classification layer, total parameters, trainable parameters, and trainable parameter names.
trainable_parameter_names = [
    name
    for name, parameter in model.named_parameters()
    if parameter.requires_grad
]

print("Model backbone:", config.BACKBONE)
print("Final classification layer:", model.fc)
print(f"Total parameters: {total_parameter_count:,}")
print(f"Trainable parameters: {trainable_parameter_count:,}")
print("Trainable parameter names:", trainable_parameter_names)

Model backbone: resnet18
Final classification layer: Linear(in_features=512, out_features=2, bias=True)
Total parameters: 11,177,538
Trainable parameters: 1,026
Trainable parameter names: ['fc.weight', 'fc.bias']


In [6]:
#Assert that the final classification layer has the correct input and output dimensions, 
# and that only the new two-class head is trainable.
assert model.fc.in_features == config.EMBEDDING_DIM
assert model.fc.out_features == config.NUM_CLASSES

assert trainable_parameter_names == [
    "fc.weight",
    "fc.bias",
]

# Assert that the number of trainable parameters is equal to the number of parameters in the final classification layer.
assert trainable_parameter_count == (
    config.EMBEDDING_DIM * config.NUM_CLASSES
    + config.NUM_CLASSES
)

print("PASS: only the new two-class head is trainable.")

PASS: only the new two-class head is trainable.


In [7]:
# Verify model input and output shapes by passing a sample batch of images through the model and checking the output shape.
sample_batch = torch.zeros(
    2,
    3,
    config.IMG_SIZE,
    config.IMG_SIZE,
)

model.eval()

with torch.no_grad():
    sample_output = model(sample_batch)

print("Input shape:", tuple(sample_batch.shape))
print("Output shape:", tuple(sample_output.shape))

assert tuple(sample_output.shape) == (
    2,
    config.NUM_CLASSES,
)

print("PASS: the model returns two logits per image.")

Input shape: (2, 3, 224, 224)
Output shape: (2, 2)
PASS: the model returns two logits per image.


### Stage 2.2 interpretation

An ImageNet-pretrained ResNet18 was configured for binary casting-defect
classification. A fixed `IMAGENET1K_V1` weight version was used to make the
pretrained starting point reproducible.

The original ResNet18 classification layer, which predicts 1,000 ImageNet
classes, was replaced with a new linear layer containing 512 input features
and two outputs. Output index 0 represents `ok_front`, while output index 1
represents the positive `def_front` class.

The pretrained backbone was frozen by setting its parameters to
`requires_grad=False`. Only `fc.weight` and `fc.bias` remained trainable. The
model contained 11,177,538 parameters in total, of which only 1,026 belonged
to the new classification head and were trainable.

A dummy batch with shape `2 × 3 × 224 × 224` produced an output with shape
`2 × 2`, confirming that each image generates two class logits. This
configuration retains the visual features learned from ImageNet while keeping
the number of trainable parameters small, which is appropriate for the
limited casting-image training dataset.

## **Stage 2.3 — Training Workflow** <font color="red">[6 marks]</font>

- **2.3.1 — Class-weighted loss + optimiser (trainable params) + seed [3]**
- **2.3.2 — Epoch loop + validation eval + early stopping [3]**

**Objective:** Train the head reproducibly with validation + early stopping.

**Implement in:** src/train.py

**Inputs → Outputs:** splits + model → trained model.pt + model_meta.json

**TODO:** class-weighted CrossEntropy + Adam over TRAINABLE params only + seed 42 (2.3.1); epoch loop with val evaluation + early stop on val-F1 (2.3.2).

**Document here:** the training summary (final val F1, epochs).

In [8]:
# TODO 2.3.1-2.3.2: !python -m src.train   (then load artifacts/model_meta.json)
# Run the reproducible training workflow.
!python -m src.train

import json
import pandas as pd

# Load the model metadata from the JSON file generated during training, which contains information about the 
# dataset version, number of training and validation images, epochs completed, best epoch, best validation F1 score, and class weights.
model_metadata = json.loads(
    config.MODEL_META_PATH.read_text(
        encoding="utf-8"
    )
)

# Create a pandas DataFrame to summarize the training metadata, making it easier to visualize and analyze the training results.
training_summary = pd.DataFrame(
    [
        {
            "dataset_version":
                model_metadata["dataset_version"],
            "training_images":
                model_metadata["training_images"],
            "validation_images":
                model_metadata["validation_images"],
            "epochs_completed":
                model_metadata["epochs_completed"],
            "best_epoch":
                model_metadata["best_epoch"],
            "best_val_f1":
                model_metadata["best_val_f1"],
            "class_weights":
                model_metadata["class_weights"],
        }
    ]
)

display(training_summary)

Dataset version: v1
Training images: 68
Validation images: 12
Class weights: [1.0, 1.0]
Trainable parameters: 1026
Epoch 1/4 | train loss: 0.6807 | train F1: 0.5152 | val loss: 0.6380 | val F1: 0.2857
Epoch 2/4 | train loss: 0.7368 | train F1: 0.2500 | val loss: 0.6116 | val F1: 0.8571
Epoch 3/4 | train loss: 0.6346 | train F1: 0.6032 | val loss: 0.6766 | val F1: 0.6667
Epoch 4/4 | train loss: 0.6626 | train F1: 0.6966 | val loss: 0.6961 | val F1: 0.6667
Best epoch: 2
Best validation F1: 0.8571
Model saved to: C:\Users\naren\Documents\MLOps-Capstone\capstone\artifacts\model.pt
Metadata saved to: C:\Users\naren\Documents\MLOps-Capstone\capstone\artifacts\model_meta.json


,dataset_version,training_images,validation_images,epochs_completed,best_epoch,best_val_f1,class_weights
0,v1,68,12,4,2,0.857143,"[1.0, 1.0]"


In [9]:
#Load the training history from the model metadata and create a pandas DataFrame 
# to visualize the training and validation loss and F1 score over epochs.
training_history = pd.DataFrame(
    model_metadata["history"]
)

display(training_history.round(4))

,epoch,train_loss,train_f1,val_loss,val_f1
0,1,0.6807,0.5152,0.6380,0.2857
1,2,0.7368,0.2500,0.6116,0.8571
2,3,0.6346,0.6032,0.6766,0.6667
3,4,0.6626,0.6966,0.6961,0.6667


In [ ]:
# Assert that the model metadata matches the expected values from the configuration, ensuring that 
# the training workflow completed successfully and that the model artifacts are present on disk.
assert model_metadata["random_seed"] == config.RANDOM_SEED
assert model_metadata["best_epoch"] >= 1
assert model_metadata["epochs_completed"] <= config.EPOCHS
assert len(model_metadata["class_weights"]) == config.NUM_CLASSES
assert config.MODEL_PATH.exists()

print("PASS: training workflow completed successfully.")

PASS: training workflow completed successfully.


### Stage 2.3 interpretation

The ResNet18 classification head was trained using dataset version `v1`,
containing 68 training images and 12 validation images. Random seed 42 was
used for Python, NumPy, PyTorch and DataLoader shuffling to support
reproducibility.

The training partition was balanced, so both `ok_front` and `def_front`
received a class weight of 1.0. Class-weighted cross-entropy was retained so
that the workflow can also handle future imbalanced dataset versions.

The Adam optimiser updated only the 1,026 trainable parameters in the new
classification head. The pretrained ResNet18 backbone remained frozen.

Training ran for four epochs. The highest validation defect F1 was 0.8571,
achieved at epoch 2. Validation F1 decreased to 0.6667 in epochs 3 and 4, so
the epoch 2 weights were restored before saving the model.

Early stopping did not activate because the configured patience was three
epochs and only two non-improving epochs followed the best result. The saved
model therefore represents the best validation checkpoint.

The variation in training and validation F1 reflects the small dataset size
and the use of random training augmentation. The validation score is used for
checkpoint selection only; final model performance will be assessed separately
on the independent test partition.

## **Stage 2.4 — MLflow Experiment Tracking** <font color="red">[6 marks]</font>

- **2.4.1 — Params + per-epoch metrics logged [3]**
- **2.4.2 — Trained model logged to the run [3]**

**Objective:** Track the experiment in MLflow.

**Implement in:** src/train.py (mlflow calls)

**Inputs → Outputs:** training run → MLflow params + per-epoch metrics + logged model

**TODO:** set_experiment; log_params; log_metrics(step=epoch) (2.4.1); log_model (2.4.2).

**Document here:** the runs table (mlflow.search_runs).

In [19]:
# TODO 2.4.1-2.4.2: mlflow.search_runs(experiment_names=[config.MLFLOW_EXPERIMENT])

import mlflow
import pandas as pd

tracking_uri = config.MLRUNS_DIR.resolve().as_uri()

# Set the MLflow tracking URI to the one specified in the configuration, which allows for logging and 
# retrieving experiment runs from the correct MLflow server or local directory.
mlflow.set_tracking_uri(
    tracking_uri
)

# Search for all runs in the specified MLflow experiment, which allows for retrieving the training results 
# and metadata for analysis and comparison.
runs = mlflow.search_runs(
    experiment_names=[
        config.MLFLOW_EXPERIMENT
    ]
)

# Filter the columns of interest from the runs DataFrame to create a summary table of the training runs,
# including run ID, status, start time, dataset version, backbone, batch size, learning rate, random seed, 
# best validation F1 score, best epoch, and epochs completed.
run_columns = [
    "run_id",
    "status",
    "start_time",
    "params.dataset_version",
    "params.backbone",
    "params.batch_size",
    "params.learning_rate",
    "params.random_seed",
    "metrics.best_val_f1",
    "metrics.best_epoch",
    "metrics.epochs_completed",
]

# Filter the run_columns list to include only those columns that are present in the runs DataFrame,
# ensuring that the summary table only contains valid columns for display.
run_columns = [
    column
    for column in run_columns
    if column in runs.columns
]

# Create a summary table of the training runs, sorted by start time in descending order,
# to display the most recent runs first, making it easier to analyze the latest training results.
runs_table = (
    runs[run_columns]
    .sort_values(
        "start_time",
        ascending=False,
    )
)

display(runs_table)

,run_id,status,start_time,params.dataset_version,params.backbone,params.batch_size,params.learning_rate,params.random_seed,metrics.best_val_f1,metrics.best_epoch,metrics.epochs_completed
0,41a1219924e14f7baaa2654c9c99aade,FINISHED,2026-08-05 15:21:17.175000+00:00,v1,resnet18,32,0.001,42,0.857143,2.0,4.0


In [20]:
#Display per-epoch training and validation metrics for the latest run in the MLflow experiment, 
# allowing for analysis of the model's performance over time.
from mlflow import MlflowClient

# Get the run ID of the latest run from the runs table, which will be used to retrieve the metric history for that specific run.
latest_run_id = runs_table.iloc[0]["run_id"]

# Create an MLflow client to interact with the MLflow tracking server, allowing for retrieval of metric history and other run information.
client = MlflowClient(
    tracking_uri=tracking_uri
)

metric_rows = []

# Retrieve the metric history for the latest run, including training and validation loss and F1 score,
# and store the metrics in a list of dictionaries for further analysis and visualization.
for metric_name in [
    "train_loss",
    "train_f1",
    "val_loss",
    "val_f1",
]:
    metric_history = client.get_metric_history(
        latest_run_id,
        metric_name,
    )

    for metric in metric_history:
        metric_rows.append(
            {
                "epoch": metric.step,
                "metric": metric_name,
                "value": metric.value,
            }
        )

# Create a pivot table from the metric rows to display the training and validation metrics for 
# each epoch in a tabular format, making it easier to analyze the model's performance over time.
metric_history_table = (
    pd.DataFrame(metric_rows)
    .pivot(
        index="epoch",
        columns="metric",
        values="value",
    )
    .reset_index()
    .round(4)
)

display(metric_history_table)

metric,epoch,train_f1,train_loss,val_f1,val_loss
0,1,0.5152,0.6807,0.2857,0.6380
1,2,0.2500,0.7368,0.8571,0.6116
2,3,0.6032,0.6346,0.6667,0.6766
3,4,0.6966,0.6626,0.6667,0.6961


In [21]:
#Verify that the logged model can be loaded from MLflow and that it produces the expected output shape when given a sample input tensor.
#set the logged model URI to the path of the model artifact in the latest run, which will be used to load the model for inference.
logged_model_uri = (
    f"runs:/{latest_run_id}/model"
)

#Load the logged model from MLflow using the specified URI and map it to the appropriate device (CPU or GPU) as defined in the configuration.
logged_model = mlflow.pytorch.load_model(
    logged_model_uri,
    map_location=config.DEVICE,
)

#Set the logged model to evaluation mode to ensure that layers like dropout and batch normalization behave correctly during inference.
logged_model.eval()

#Create a sample input tensor of zeros with the appropriate shape (batch size, channels, height, width) to test the logged model's output shape.
test_input = torch.zeros(
    1,
    3,
    config.IMG_SIZE,
    config.IMG_SIZE,
)

# Test the logged model with the sample input.
with torch.no_grad():
    test_output = logged_model(test_input)

print("Logged model URI:", logged_model_uri)
print("Output shape:", tuple(test_output.shape))

# Assert that the output shape of the logged model matches the expected shape (batch size, number of classes),
# ensuring that the model was loaded correctly and is functioning as expected.
assert tuple(test_output.shape) == (
    1,
    config.NUM_CLASSES,
)

print("PASS: MLflow model loaded successfully.")

Logged model URI: runs:/41a1219924e14f7baaa2654c9c99aade/model
Output shape: (1, 2)
PASS: MLflow model loaded successfully.


### Stage 2.4 interpretation

The training workflow was tracked in the local MLflow experiment
`casting_defect_detection`. The run recorded the dataset version, model
configuration, class weights, optimiser settings, training and validation
sizes, random seed and early-stopping configuration.

Training loss, training defect F1, validation loss and validation defect F1
were logged for each of the four epochs. The tracked results reproduced the
earlier training run, with the highest validation defect F1 of 0.8571 achieved
at epoch 2.

After training, the best validation checkpoint was restored and logged as an
MLflow PyTorch model. The model was stored under run ID
`41a1219924e14f7baaa2654c9c99aade` and can be referenced using:

`runs:/41a1219924e14f7baaa2654c9c99aade/model`

The logged model was successfully reloaded through MLflow and produced an
output tensor with shape `(1, 2)`, confirming that the artifact is valid and
returns two class logits.

The reproducibility controls were also effective, as the tracked run produced
the same epoch-level results and best checkpoint as the previous Stage 2.3
execution.

## **Stage 3.1 — Evaluation & Failure Analysis** <font color="red">[7 marks]</font>

- **3.1.1 — Imbalance-aware metrics + model selection [4]**
- **3.1.2 — Confusion/ROC plots + failure-case analysis [3]**

**Objective:** Evaluate on defect recall/F1, not accuracy.

**Implement in:** src/evaluate.py

**Inputs → Outputs:** model + test split → metrics.json + model_eval.png

**TODO:** report recall(defect)/precision/F1/ROC-AUC + select on recall/F1 (3.1.1); plot ROC + confusion and inspect misclassified samples (3.1.2).

**Document here:** the metrics + a short interpretation (why recall matters here).

In [ ]:
# TODO 3.1.1-3.1.2: load artifacts/metrics.json + show artifacts/model_eval.png


## **Stage 3.2 — Model Registry & Promotion** <font color="red">[5 marks]</font>

- **3.2.1 — Best model registered in MLflow Registry [2]**
- **3.2.2 — Promotion to production alias + version history [3]**

**Objective:** Register and promote the best model.

**Implement in:** src/train.py (MLflow registry)

**Inputs → Outputs:** best run → registered model + @production alias

**TODO:** register_model(casting_defect_classifier) (3.2.1); set_registered_model_alias('production', v) with version history (3.2.2).

**Document here:** the registry version + alias (screenshot the MLflow Models page).

In [ ]:
# TODO 3.2.1-3.2.2: MlflowClient().get_model_version_by_alias(config.REGISTERED_MODEL, 'production')


## **Stage 3.3 — FastAPI Inference Service** <font color="red">[6 marks]</font>

- **3.3.1 — Image /predict + /health + logging + validation [4]** — *to be done in app.py*
- **3.3.2 — Live call demonstrated in-notebook [2]** — *to be done in this notebook*

**Objective:** Serve the model behind an image API and demonstrate it.

**Implement in:** app.py (+ a TestClient demo here)

**Inputs → Outputs:** image upload → {label, prob_defect, confidence}; /health; predictions logged

**TODO:** implement /health + POST /predict (UploadFile) with validation (bad image 400, no-model 503) + logging (3.3.1, in app.py); demo with TestClient on a sample image (3.3.2, here).

**Document here:** the live request/response (TestClient output).

In [ ]:
# TODO 3.3.2: with TestClient(app.app) as c: call /health and POST a sample image to /predict


## **Stage 3.4 — Containerisation & CI/CD** <font color="red">[7 marks]</font>

- **3.4.1 — Valid Dockerfile [3]** — *to be done in this notebook and the report*
- **3.4.2 — GitHub Actions CI + passing-test evidence [4]** — *to be done in this notebook and the report*

**Objective:** Containerise and automate tests.

**Implement in:** Dockerfile + .github/workflows/ci.yml (provided — review + evidence them)

**Inputs → Outputs:** code → Docker image; push/PR → CI runs pytest + docker build

**TODO:** show the Dockerfile (3.4.1); run pytest here + include a screenshot of the green CI run and successful docker build (3.4.2).

**Document here:** pytest output + Docker/CI screenshots **(Note: Don't add a repo link)**

In [ ]:
# TODO 3.4.1-3.4.2: print Dockerfile + ci.yml; run pytest; embed your CI/Docker screenshots
